# 📊 Data Understanding — IndoToxic2024 Dataset
**Proyek:** Indonesian Hate Speech Analyzer (Kelompok 6)  
**Arsitektur Model:** XLM-RoBERTa  
**Notebook ini** melakukan Exploratory Data Analysis (EDA) pada dataset `indotoxic2024_annotated_data_v2_final.csv`.

---
## Bab 1: Setup & Data Loading

In [ ]:
# === Import Library ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_colwidth', 150)

print("✅ Library berhasil dimuat.")

### 1.1 Memuat Dataset Utama

In [ ]:
import pandas as pd
from pathlib import Path

# Menentukan root directory secara otomatis & dinamis (jalan di semua laptop)
# Cek apakah working directory saat ini ada di folder 'notebooks'
current_dir = Path.cwd()
if current_dir.name == 'notebooks':
    project_root = current_dir.parent # Naik 1 level ke root
else:
    project_root = current_dir # Asumsikan sudah berada di root

# Menyusun path menuju file CSV data raw
file_path = project_root / 'data' / 'raw' / 'indotoxic2024_annotated_data_v2_final.csv'

# Membaca dataset utama dari CSV
df = pd.read_csv(file_path)

print(f"Jumlah Baris (Data Point) : {df.shape[0]:,}")
print(f"Jumlah Kolom (Variabel)   : {df.shape[1]}")
print()
df.head()


### 1.2 Memahami Struktur Dataset (Poin 1)
Menampilkan nama kolom, tipe data, dan jumlah data non-null untuk setiap variabel.

In [ ]:
df.info()

In [ ]:
# Daftar nama kolom beserta tipe datanya
print("=== Daftar Kolom & Tipe Data ===")
for i, (col, dtype) in enumerate(zip(df.columns, df.dtypes), 1):
    print(f"  {i:2d}. {col:<40s} → {dtype}")

### 1.3 Memeriksa Kualitas Data (Poin 2)
Mengecek data kosong (NaN/missing values), data duplikat, dan distribusi panjang teks.

In [ ]:
# --- Cek Missing Values ---
print("=== Missing Values per Kolom ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Jumlah NaN': missing, 'Persentase (%)': missing_pct})
print(missing_df[missing_df['Jumlah NaN'] > 0].to_string())
print()

# --- Cek Duplikat ---
n_dup = df.duplicated(subset='text').sum()
print(f"Jumlah teks duplikat: {n_dup}")
print()

# --- Statistik Panjang Teks ---
df['char_count'] = df['text'].astype(str).str.len()
df['word_count'] = df['text'].astype(str).str.split().str.len()

print("=== Statistik Panjang Teks ===")
print(df[['char_count', 'word_count']].describe().round(1).to_string())

In [ ]:
# --- Identifikasi Teks Sangat Pendek & Sangat Panjang ---
very_short = df[df['word_count'] <= 3]
very_long  = df[df['word_count'] >= 200]

print(f"Teks sangat pendek (≤ 3 kata) : {len(very_short):,} baris")
print(f"Teks sangat panjang (≥ 200 kata): {len(very_long):,} baris")
print()

if len(very_short) > 0:
    print("--- Sampel Teks Sangat Pendek ---")
    print(very_short[['text_id', 'text']].head(5).to_string(index=False))

---
## Bab 2: Parsing & Agregasi Label (Majority Voting)

Kolom-kolom label pada dataset masih tersimpan sebagai **string representasi list** (contoh: `"['1', '0']"`).  
Kita perlu mengubahnya menjadi satu label konsensus menggunakan rumus **Majority Voting**:

$$\text{Label} = \begin{cases} 1, & \text{jika rata-rata vote} > 0.5 \\ 0, & \text{jika rata-rata vote} < 0.5 \\ \text{Disagreement}, & \text{jika rata-rata vote} = 0.5 \end{cases}$$

In [ ]:
# === Fungsi Parsing Anotasi List ===

def parse_annotation_list(annotation_str):
    """Mengubah string list anotasi menjadi list angka.
    Contoh: "['1', '0']" -> [1, 0]
    """
    try:
        parsed = ast.literal_eval(annotation_str)
        return [int(x) for x in parsed]
    except (ValueError, SyntaxError):
        return []

print("Fungsi parse_annotation_list() berhasil didefinisikan.")

### 2.1 Analisis Distribusi Jumlah Anotator per Teks
Sebelum melakukan majority voting, kita periksa apakah jumlah annotator di dalam list selalu 2, atau ada teks yang dinilai oleh 1, 3, atau lebih annotator.

In [ ]:
# Menghitung panjang list annotators_id untuk setiap baris
df['num_annotators'] = df['annotators_id'].apply(lambda x: len(parse_annotation_list(x)))

# Tabel frekuensi jumlah annotator
annotator_counts = df['num_annotators'].value_counts().sort_index()
annotator_pct = (annotator_counts / len(df) * 100).round(2)

dist_annotator_df = pd.DataFrame({
    'Jumlah Anotator (Panjang List)': annotator_counts.index,
    'Jumlah Teks (Baris)': annotator_counts.values,
    'Persentase (%)': annotator_pct.values
})

print("=== Distribusi Jumlah Anotator per Teks ===")
print(dist_annotator_df.to_string(index=False))
print()
print(f"Total baris data         : {len(df):,}")
print(f"Rentang jumlah annotator : {df['num_annotators'].min()} s/d {df['num_annotators'].max()} annotator per teks")

In [ ]:
# Visualisasi Distribusi Jumlah Anotator
fig, ax = plt.subplots(figsize=(10, 5))

# Kelompokkan kategori: 1 sampai 6, dan >6
counts_display = annotator_counts[annotator_counts.index <= 6].copy()
other_count = annotator_counts[annotator_counts.index > 6].sum()
if other_count > 0:
    counts_display['>6'] = other_count

bars = ax.bar([str(idx) for idx in counts_display.index], counts_display.values, color='#3498db', edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, counts_display.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('Distribusi Jumlah Anotator per Teks (Panjang List)', fontsize=13, fontweight='bold')
ax.set_xlabel('Jumlah Anotator yang Menilai')
ax.set_ylabel('Jumlah Teks')
plt.tight_layout()
plt.show()

### 💡 Temuan Analisis Anotator:
Dari data di atas, kita menemukan fakta empiris penting:
1. **Jumlah annotator TIDAK HANYA 2 orang!** Rentang annotator berkisar dari **1 hingga 19 orang** per teks.
2. **55.4% (15.748 baris)** ternyata hanya dinilai oleh **1 annotator**.
3. **27.8% (7.907 baris)** dinilai oleh **2 annotator**.
4. **16.8% (4.793 baris)** dinilai oleh **3 atau lebih annotator**.

Ini membuktikan mengapa metode **Majority Voting** (perhitungan rata-rata suara > 0.5) mutlak diperlukan, karena sistem harus mampu menangani jumlah voter yang dinamis (1, 2, 3, hingga 19 orang)!

In [ ]:
# === Fungsi Majority Voting ===

def majority_vote(annotation_str):
    """Menghitung label konsensus dari string list anotasi.
    Returns:
        1   -> Mayoritas positif (rata-rata > 0.5)
        0   -> Mayoritas negatif (rata-rata < 0.5)
        0.5 -> Disagreement / Tie (rata-rata = 0.5)
    """
    votes = parse_annotation_list(annotation_str)
    if len(votes) == 0:
        return np.nan
    avg = np.mean(votes)
    if avg > 0.5:
        return 1
    elif avg < 0.5:
        return 0
    else:
        return 0.5  # Disagreement

print("Fungsi majority_vote() berhasil didefinisikan.")

In [ ]:
# === Kolom-kolom label yang akan diproses ===
label_columns = [
    'toxicity',
    'identity_attack',
    'threat_incitement_to_violence',
    'insults',
    'profanity_obscenity',
    'sexually_explicit',
    'polarized',
    'related_to_election_2024',
    'is_noise_or_spam_text'
]

# Menerapkan majority voting ke semua kolom label
for col in label_columns:
    new_col = f'{col}_label'
    df[new_col] = df[col].apply(majority_vote)
    print(f"  ✅ {col} → {new_col}")

print()
print("Semua kolom label berhasil diparsing!")
print()

# Menampilkan sampel hasil parsing
sample_cols = ['text_id', 'toxicity', 'toxicity_label', 'insults', 'insults_label']
df[sample_cols].head(5)

---
## Bab 3: Analisis Distribusi Label & Pembuktian Statistik Kritis (Poin 3)

### 3.1 Distribusi Label Toksisitas (Task Utama: Biner)

In [ ]:
# === Distribusi Toxicity (Biner) ===
tox_counts = df['toxicity_label'].value_counts().sort_index()

label_map = {0: 'Non-Toxic (0)', 0.5: 'Disagreement (0.5)', 1: 'Toxic (1)'}
tox_display = tox_counts.rename(index=label_map)

print("=== Distribusi Label Toksisitas ===")
for label, count in tox_display.items():
    pct = count / len(df) * 100
    print(f"  {label:<25s}: {count:>6,} ({pct:.1f}%)")

# Visualisasi Bar Chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ecc71', '#f39c12', '#e74c3c']
bars = ax.bar(tox_display.index, tox_display.values, color=colors, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, tox_display.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontweight='bold')

ax.set_title('Distribusi Label Toksisitas (Majority Voting)', fontsize=14, fontweight='bold')
ax.set_ylabel('Jumlah Data')
ax.set_xlabel('Kategori Label')
plt.tight_layout()
plt.show()

### 3.2 Distribusi Sub-Kategori Multi-Label

In [ ]:
# === Distribusi 5 Sub-Kategori Toksisitas ===
multi_label_cols = [
    'identity_attack_label',
    'threat_incitement_to_violence_label',
    'insults_label',
    'profanity_obscenity_label',
    'sexually_explicit_label'
]

multi_label_names = [
    'Identity Attack\n(SARA)',
    'Threat\n(Ancaman)',
    'Insults\n(Hinaan)',
    'Profanity\n(Kata Kasar)',
    'Sexually\nExplicit'
]

# Menghitung jumlah kelas 1 (positif) untuk setiap sub-kategori
positive_counts = [df[col].apply(lambda x: 1 if x == 1 else 0).sum() for col in multi_label_cols]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(multi_label_names, positive_counts, color='#e74c3c', edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, positive_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val:,}', ha='center', va='bottom', fontweight='bold')

ax.set_title('Jumlah Teks Positif (Kelas 1) per Sub-Kategori Toksisitas', fontsize=13, fontweight='bold')
ax.set_ylabel('Jumlah Data Berlabel 1')
ax.set_xlabel('Sub-Kategori')
plt.tight_layout()
plt.show()

### 3.3 Rekonstruksi Tabel Statistik 
Memvalidasi angka-angka di **Tabel 5: Distribusi & Statistik Kritis Variabel** pada dokumen `Data_Understanding_Guide.md`.

In [ ]:
# === Rekonstruksi Tabel Statistik Kritis ===
# Memvalidasi angka dari Data_Understanding_Guide.md

all_label_cols = [
    ('toxicity_label',                         'toxicity (Task Utama)'),
    ('polarized_label',                        'polarized'),
    ('identity_attack_label',                  'identity_attack'),
    ('insults_label',                          'insults'),
    ('profanity_obscenity_label',              'profanity_obscenity'),
    ('threat_incitement_to_violence_label',     'threat_incitement_to_violence'),
    ('sexually_explicit_label',                'sexually_explicit'),
    ('is_noise_or_spam_text_label',            'is_noise_or_spam_text'),
    ('related_to_election_2024_label',         'related_to_election_2024'),
]

rows = []
for col, name in all_label_cols:
    total = len(df)
    n_0   = (df[col] == 0).sum()
    n_1   = (df[col] == 1).sum()
    n_dis = (df[col] == 0.5).sum()
    
    # Hitung rasio imbalance (Kelas 0 : Kelas 1)
    ratio = f"1 : {int(round(n_0 / n_1))}" if n_1 > 0 else "N/A"
    
    rows.append({
        'Variabel': name,
        'Kelas 0': f"{n_0:,} ({n_0/total*100:.1f}%)",
        'Kelas 1': f"{n_1:,} ({n_1/total*100:.1f}%)",
        'Disagreement': f"{n_dis:,} ({n_dis/total*100:.1f}%)",
        'Rasio Imbalance': ratio
    })

stats_df = pd.DataFrame(rows)
print("=" * 100)
print("TABEL STATISTIK ")
print("=" * 100)
print(stats_df.to_string(index=False))

---
## Bab 4: Analisis Karakteristik Teks & Pembuktian Tantangan Data (Poin 4 & 6)

### 4.1 Distribusi Panjang Teks (Karakter & Kata)

In [ ]:
# === Histogram Panjang Teks: Toxic vs Non-Toxic ===
df_toxic     = df[df['toxicity_label'] == 1]
df_nontoxic  = df[df['toxicity_label'] == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram Jumlah Kata
axes[0].hist(df_nontoxic['word_count'], bins=50, alpha=0.6, label='Non-Toxic', color='#2ecc71', edgecolor='black', linewidth=0.3)
axes[0].hist(df_toxic['word_count'],    bins=50, alpha=0.6, label='Toxic',     color='#e74c3c', edgecolor='black', linewidth=0.3)
axes[0].set_title('Distribusi Jumlah Kata', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Jumlah Kata')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()

# Histogram Jumlah Karakter
axes[1].hist(df_nontoxic['char_count'], bins=50, alpha=0.6, label='Non-Toxic', color='#2ecc71', edgecolor='black', linewidth=0.3)
axes[1].hist(df_toxic['char_count'],    bins=50, alpha=0.6, label='Toxic',     color='#e74c3c', edgecolor='black', linewidth=0.3)
axes[1].set_title('Distribusi Jumlah Karakter', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Jumlah Karakter')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.suptitle('Perbandingan Panjang Teks: Toxic vs Non-Toxic', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Statistik rata-rata
print("=== Rata-rata Panjang Teks ===")
print(f"  Non-Toxic → Kata: {df_nontoxic['word_count'].mean():.1f}, Karakter: {df_nontoxic['char_count'].mean():.1f}")
print(f"  Toxic     → Kata: {df_toxic['word_count'].mean():.1f}, Karakter: {df_toxic['char_count'].mean():.1f}")

### 4.2 Kata yang Paling Sering Muncul pada Teks Toxic

In [ ]:
# === Top 20 Kata Paling Sering Muncul di Teks Toxic ===
from collections import Counter

# Mengambil semua kata dari teks berlabel Toxic
all_words_toxic = ' '.join(df_toxic['text'].astype(str).str.lower()).split()

# Menghitung frekuensi kemunculan
word_freq = Counter(all_words_toxic)
top_20 = word_freq.most_common(20)

# Visualisasi Horizontal Bar Chart
words, counts = zip(*top_20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(words)), counts, color='#e74c3c', edgecolor='black', linewidth=0.3)
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words)
ax.invert_yaxis()
ax.set_title('Top 20 Kata Paling Sering Muncul di Teks Toxic', fontsize=13, fontweight='bold')
ax.set_xlabel('Frekuensi Kemunculan')
plt.tight_layout()
plt.show()

### 4.3 Word Cloud — Teks Toxic

In [ ]:
%pip install wordcloud

In [ ]:
# === Word Cloud: Kata Dominan pada Teks Toxic (Diperbarui) ===
import re

try:
    from wordcloud import WordCloud

    # 1. Menggabungkan semua teks menjadi huruf kecil
    text_toxic_all = ' '.join(df_toxic['text'].astype(str).str.lower())
    
    # 2. Pembersihan ringan: Hapus URL dan @mention agar Word Cloud bersih
    text_toxic_all = re.sub(r'http\S+|www\S+|https\S+', '', text_toxic_all, flags=re.MULTILINE)
    text_toxic_all = re.sub(r'\@\w+', '', text_toxic_all)
    
    # 3. Mendefinisikan kata-kata umum yang tidak penting (Stopwords Bahasa Indonesia & Slang)
    indo_stopwords = set([
        'yang', 'di', 'dan', 'ini', 'itu', 'untuk', 'dengan', 'dari', 'ke', 'pada', 
        'dalam', 'adalah', 'juga', 'tidak', 'ada', 'orang', 'yg', 'ya', 'aja', 
        'kalo', 'buat', 'sama', 'bisa', 'karena', 'kalau', 'akan', 'aku', 'saya',
        'dia', 'mereka', 'kita', 'kamu', 'udah', 'gak', 'nya', 'kok', 'sih', 'lagi',
        'lebih', 'banyak', 'sudah', 'baru', 'jadi'
    ])

    # 4. Generate Word Cloud dengan memasukkan parameter stopwords
    wordcloud = WordCloud(
        width=1000, height=500,
        background_color='white',
        colormap='Reds',      # Warna merah cocok untuk konteks bahaya/toxic
        max_words=100,        # Kurangi jadi 100 agar kata besar lebih jelas terbaca
        stopwords=indo_stopwords, # <--- KUNCI UTAMANYA DI SINI
        collocations=False
    ).generate(text_toxic_all)

    # 5. Visualisasi
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(wordcloud, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Word Cloud — Kata Dominan pada Teks Berlabel Toxic', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("⚠️ Library 'wordcloud' belum terinstal.")
    print("   Jalankan: pip install wordcloud")

In [ ]:
import re
import matplotlib.pyplot as plt

# 1. Pengecekan dan instalasi otomatis untuk NLTK
try:
    import nltk
    from nltk.corpus import stopwords
except ImportError:
    print("Modul 'nltk' belum terinstal. Memulai instalasi...")
    %pip install nltk
    import nltk
    from nltk.corpus import stopwords

# 2. Pengecekan dan instalasi otomatis untuk WordCloud
try:
    from wordcloud import WordCloud
except ImportError:
    print("Modul 'wordcloud' belum terinstal. Memulai instalasi...")
    %pip install wordcloud
    from wordcloud import WordCloud


# --- PROSES PEMBUATAN WORDCLOUD ---

# Download database stopwords dari NLTK (Hanya akan di-download jika belum ada)
nltk.download('stopwords', quiet=True)

# Ambil list stopwords resmi bahasa Indonesia dari NLTK
nltk_stopwords = set(stopwords.words('indonesian'))

# TAMBAHKAN slang/kata gaul secara manual karena NLTK tidak tahu kata gaul
slang_stopwords = {'yg', 'aja', 'kalo', 'buat', 'sama', 'udah', 'gak', 'nya', 'kok', 'sih', 'nih', 'tuh'}

# Gabungkan keduanya
final_stopwords = nltk_stopwords.union(slang_stopwords)

# Preprocessing teks dasar
text_toxic_all = ' '.join(df_toxic['text'].astype(str).str.lower())
text_toxic_all = re.sub(r'http\S+|www\S+|https\S+', '', text_toxic_all)
text_toxic_all = re.sub(r'\@\w+', '', text_toxic_all)

# Masukkan final_stopwords ke dalam WordCloud
wordcloud = WordCloud(
    width=1000, height=500,
    background_color='white',
    colormap='Reds',
    max_words=100,
    stopwords=final_stopwords, # Gunakan gabungan NLTK + Manual
    collocations=False
).generate(text_toxic_all)

# Visualisasi
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wordcloud, interpolation='bilinear')
ax.axis('off')
plt.tight_layout()
plt.show()


### 4.4 Heatmap Korelasi Antar Sub-Label
Melihat apakah kalimat hinaan (*insults*) sering muncul bersamaan dengan kata kasar (*profanity*), sebagai justifikasi pemilihan fungsi aktivasi **Sigmoid** (independen per label).

In [ ]:
# === Heatmap Korelasi Antar Sub-Label ===
corr_cols = [
    'toxicity_label',
    'identity_attack_label',
    'threat_incitement_to_violence_label',
    'insults_label',
    'profanity_obscenity_label',
    'sexually_explicit_label'
]

corr_names = ['Toxicity', 'SARA', 'Ancaman', 'Hinaan', 'Kata Kasar', 'Seksual']

# Hanya ambil data yang bukan disagreement (0 atau 1)
df_corr = df[corr_cols].copy()
df_corr = df_corr[(df_corr != 0.5).all(axis=1)]

correlation = df_corr.corr()
correlation.index = corr_names
correlation.columns = corr_names

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='RdYlGn_r',
            vmin=-0.1, vmax=1, linewidths=0.5, ax=ax,
            square=True)
ax.set_title('Heatmap Korelasi Antar Label Toksisitas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Bab 5: Kesimpulan & Rekomendasi (Poin 7)

Berdasarkan eksplorasi data di atas, berikut adalah temuan utama dan rekomendasi untuk tahap selanjutnya:

### 📌 Temuan Utama
1. **Ketimpangan Kelas Sangat Ekstrem:** Label `toxicity` memiliki rasio Non-Toxic vs Toxic sekitar 11:1. Sub-kategori `sexually_explicit` bahkan mencapai rasio 566:1. Model baseline tanpa penanganan *imbalance* hampir pasti akan gagal mendeteksi kelas minoritas.
2. **Disagreement (Ambiguitas) Signifikan:** Sekitar 6.5% data pada label `toxicity` memiliki nilai *tie* (0.5) di mana para annotator tidak sepakat. Data ini perlu strategi penanganan khusus (drop, pisahkan, atau gunakan sebagai *soft label*).
3. **Korelasi Antar Label:** Heatmap korelasi menunjukkan bahwa beberapa sub-label (seperti `insults` dan `profanity`) memiliki korelasi positif. Ini menjustifikasi penggunaan **Sigmoid** (bukan Softmax) pada arsitektur Multi-Label, karena satu teks bisa memiliki lebih dari satu label aktif secara bersamaan.

### 🔧 Rekomendasi untuk Tahap Data Engineering
1. **Penanganan Imbalance:** Wajib menerapkan teknik seperti *Class Weights*, *Focal Loss*, atau *oversampling* pada kelas minoritas.
2. **Text Cleaning:** Perlu dilakukan normalisasi slang, penghapusan URL/mention, dan pembersihan karakter khusus sebelum tokenisasi.
3. **Strategi Disagreement:** Tentukan apakah data dengan label 0.5 akan di-drop, dibulatkan, atau digunakan sebagai *soft label* untuk training.
4. **Analisis per Topik:** Pertimbangkan untuk menganalisis distribusi toksisitas berdasarkan kolom `topic` untuk memahami domain mana yang paling banyak mengandung *hate speech*.

---
## Bab 6: Ekspor Data Interim

In [ ]:
# ============================================================
# KODE EKSPOR DATA INTERIM
# Uncomment (hapus tanda #) ketika siap melanjutkan ke
# tahap Data Engineering / Cleaning.
# ============================================================

# import os
# 
# # Pilih kolom yang relevan untuk disimpan
# export_cols = [
#     'text_id', 'text', 'initial_paragraph', 'topic',
#     'toxicity_label', 'identity_attack_label',
#     'threat_incitement_to_violence_label', 'insults_label',
#     'profanity_obscenity_label', 'sexually_explicit_label',
#     'polarized_label', 'related_to_election_2024_label',
#     'is_noise_or_spam_text_label',
#     'char_count', 'word_count', 'num_annotators'
# ]
# 
# df_export = df[export_cols].copy()
# 
# # Buat folder jika belum ada
# os.makedirs('data/interim', exist_ok=True)
# 
# # Simpan ke CSV
# df_export.to_csv('data/interim/data_parsed.csv', index=False)
# print(f"Data berhasil disimpan ke data/interim/data_parsed.csv")
# print(f"   Jumlah baris: {len(df_export):,}")
# print(f"   Jumlah kolom: {len(df_export.columns)}")